# 01 — Data and Design

Checks the panel and the research design before anything is estimated.

The questions here are whether the ratifying countries and the control group are
comparable, how much of the result rests on any single control country, how
large a change the data could detect, and whether an outcome reacts to a
ratification date that has been deliberately set too early. These checks come
first because they can rule an outcome out of the causal analysis altogether.

The notebook ends with a table of which methods each outcome supports, which
notebook 02 reads.

**Input** `data/gbv_panel_analysis.csv`, built by `src/build_analysis_panel.py`.
**Output** `outputs/diagnostics/`, and one figure in `outputs/figures/`.

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "src"))

from analysis_helpers import *          # noqa: F401,F403
import estimators as E
import scm as SCM

DATA_PATH = resolve_data_path(PROJECT_DIR)
OUT = make_output_dirs(PROJECT_DIR)
_d = load_and_prepare_data(DATA_PATH, OUT)
df, SAMPLES = _d["df"], _d["SAMPLES"]
country_info, inventory = _d["country_info"], _d["inventory"]
ALL_CTRL = _d["ALL_CTRL"]

## 1. Panel integrity

Dimensions, coverage, duplicate country-years, and the source of every
column that was added or derived when the analysis panel was built.

In [ ]:
print("rows:", len(df), "| countries:", df.country.nunique(),
      "| years:", df.year.min(), "-", df.year.max())
print("duplicate country-year rows:", int(df.duplicated(["country","year"]).sum()))
print("ratifiers:", int(country_info.convention_ratified_year.notna().sum()),
      "| ratification years:", int(country_info.convention_ratified_year.min()),
      "-", int(country_info.convention_ratified_year.max()))
print("countries that never ratified:", sorted(country_info.loc[country_info.convention_ratified_year.isna(),"country"]))
print("\nvariables added or derived when the panel was built:")
sources = pd.read_csv(PROJECT_DIR / "data" / "variable_sources.csv")
print(sources[["variable", "source"]].to_string(index=False))

## 2. Outcome data quality

Rates of exactly zero, large year-on-year discontinuities, and gaps inside a
country's own series. A rate of exactly zero in a small country reflects a count
of zero rather than an absence of violence, and a jump of more than about 1.8x
on a non-trivial base is more often a change in recording practice than a change
in behaviour. Both matter because the outcome variables are administrative
crime statistics, not survey measures of incidence.

In [ ]:
qual = []
for key in ["fhr", "male_hom", "fem_ipf", "svr", "rape"]:
    s = df[["country","year",key]].dropna()
    zeros = int((s[key] == 0).sum())
    brk = 0
    for c, g in s.sort_values("year").groupby("country"):
        g = g.reset_index(drop=True)
        for i in range(1, len(g)):
            if g.year[i] - g.year[i-1] == 1 and g[key][i-1] > 0.2:
                r = g[key][i] / g[key][i-1]
                if r > 1.8 or r < 1/1.8:
                    brk += 1
    gaps = 0
    for c, g in s[s.year >= 2000].groupby("country"):
        yrs = sorted(g.year)
        gaps += len([y for y in range(yrs[0], yrs[-1]+1) if y not in yrs])
    qual.append({"outcome": key, "n_obs": len(s), "countries": s.country.nunique(),
                 "exact_zeros": zeros, "yoy_breaks_gt1.8x": brk,
                 "internal_missing_years_2000plus": gaps})
qual_df = pd.DataFrame(qual)
print(qual_df.to_string(index=False))
qual_df.to_csv(OUT["diagnostics"] / "outcome_data_quality.csv", index=False)

In [ ]:
# Turkey specifically: the series break that every Turkey result depends on.
tur = df[(df.country=="Turkey") & df.fhr.notna()][["year","fhr"]]
print("Turkey female homicide, observed years only:")
print(tur.to_string(index=False))
missing = [y for y in range(int(tur.year.min()), int(tur.year.max())+1)
           if y not in set(tur.year)]
print("\nMISSING YEARS INSIDE TURKEY'S RANGE:", missing)
print("Turkey ratified 2012, entry into force 2014, withdrew 2021.")
print("The 2013-2014 gap sits exactly between ratification and entry into force,")
print("and the level drops 1.617 (2012) -> 1.032 (2015), a 36% fall across the gap.")
print("Any Turkey result must state that this may be a recording change.")

## 3. Treatment timing and the composition of the control group

Ratification is staggered across 2012–2024. Under two-way fixed effects the
comparison pool in year *t* consists of never-treated, not-yet-treated and
already-treated countries. Comparisons that use already-treated units as
controls are the source of the bias described by Goodman-Bacon (2021), and they
become more important as the clean comparison pool shrinks. This table shows how
quickly that happens.

In [ ]:
d = SAMPLES["fhr"]
tab = d.groupby("year").apply(lambda g: pd.Series({
    "countries": g.country.nunique(),
    "already_treated": int(((g.treated_ever==1)&(g.post_ratification==1)).sum()),
    "not_yet_treated": int(((g.treated_ever==1)&(g.post_ratification==0)).sum()),
    "never_treated": int((g.treated_ever==0).sum())}), include_groups=False)
print(tab.to_string())
tab.to_csv(OUT["diagnostics"] / "treatment_timing_by_year.csv")
print("\nCohorts:", sorted(d.loc[d.treated_ever==1,"convention_ratified_year"].dropna().unique().astype(int)))

In [ ]:
# Adoption timing and data coverage. Each marker is one observed country-year,
# so gaps in a country's series are visible rather than smoothed over by a
# spanning bar. Countries are ordered by ratification year.
d = SAMPLES["fhr"]
info = (d[["country", "convention_ratified_year", "treated_ever"]]
        .drop_duplicates("country").copy())
info["sort_year"] = info["convention_ratified_year"].fillna(9999)
info = info.sort_values(["sort_year", "country"], ascending=[False, False])
ypos = {c: i for i, c in enumerate(info.country)}

fig, ax = plt.subplots(figsize=(10, 11))
obs = d[["country", "year", "post_ratification"]].copy()
obs["y"] = obs.country.map(ypos)
pre = obs[obs.post_ratification.eq(0)]
post = obs[obs.post_ratification.eq(1)]
ax.scatter(pre.year, pre.y, s=16, marker="s", color="#BBBBBB",
           label="observed, not yet ratified")
ax.scatter(post.year, post.y, s=16, marker="s", color=C_TREATED,
           label="observed, ratified")

for _, r in info.iterrows():
    if r.treated_ever == 1 and pd.notna(r.convention_ratified_year):
        ax.plot(int(r.convention_ratified_year), ypos[r.country], "|",
                color=C_POLICY, markersize=13, markeredgewidth=2.2)

ax.set_yticks(range(len(info)))
ax.set_yticklabels(info.country, fontsize=8.5)
ax.set_xlabel("Year", labelpad=8)
ax.set_ylim(-1, len(info))
ax.set_title("Ratification timing and data coverage, 2000-2023"
             f"\n{int((info.treated_ever == 1).sum())} treated countries, "
             f"{int((info.treated_ever == 0).sum())} untreated within the window",
             fontsize=13)
ax.grid(axis="x", alpha=.3)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.075), ncol=2,
          fontsize=9, frameon=False)
fig_note("Red tick marks ratification. Countries ratifying after their last "
         "observed year contribute no post-treatment data. Sample: female "
         "homicide, 2000-2023, microstates excluded. Source: UNODC UN-CTS.")
save_fig(OUT["figures"], "ratification_timing.png")

no_post = [c for c in info.country
           if d.loc[d.country.eq(c), "post_ratification"].max() == 0
           and info.loc[info.country.eq(c), "treated_ever"].iloc[0] == 1]
print(f"Treated countries with no post-ratification observations: {no_post}")

## 4. Comparability of treated and control countries

This is the binding constraint on the project. Seven countries are untreated
within the estimation window, and all six that never ratify are post-socialist
countries, while the treated group is dominated by Western Europe. The comparison
therefore mixes a treatment contrast with a regional one.

The leave-one-control-out exercise below asks how much of the estimate rests on
any single control country.

In [ ]:
bal = pretreatment_balance(SAMPLES["fhr"], "fhr", cutoff=2013)
ctl = bal[bal.treated==0].sort_values("pre_mean", ascending=False)
trt = bal[bal.treated==1]
print("CONTROL COUNTRIES (pre-2013):")
print(ctl.to_string(index=False))
print(f"\nTreated pre-mean: median={trt.pre_mean.median():.3f} "
      f"[{trt.pre_mean.min():.3f}, {trt.pre_mean.max():.3f}]")
print(f"Treated pre-slope: median={trt.pre_slope.median():.4f}")
print(f"\nMean GDP p.c.  treated={trt.gdp_per_capita_constant_2015_usd.mean():,.0f}  "
      f"control={ctl.gdp_per_capita_constant_2015_usd.mean():,.0f}")
print(f"post-socialist share  treated="
      f"{df[df.treated_ever==1].drop_duplicates('country').post_socialist.mean():.2f}  "
      f"control={df[df.treated_ever==0].drop_duplicates('country').post_socialist.mean():.2f}")
bal.to_csv(OUT["diagnostics"] / "pretreatment_balance.csv", index=False)

In [ ]:
# Leave-one-CONTROL-out. A design finding, not a robustness check.
rows = []
base = E.feols(SAMPLES["fhr"], "fhr", ["did_interaction"], ["country","year"],
               cluster="country", name="baseline")
rows.append(E.coef_row(base, "did_interaction", "Baseline (all controls)"))
for c in ALL_CTRL:
    sub = SAMPLES["fhr"][SAMPLES["fhr"].country != c]
    r = E.feols(sub, "fhr", ["did_interaction"], ["country","year"],
                cluster="country", name=f"excl {c}")
    rows.append(E.coef_row(r, "did_interaction", f"Excluding {c}"))
loo = export_table(rows, OUT["diagnostics"] / "leave_one_control_out.csv",
                   "leave-one-control-out")
print(loo[["label","b","se","p","N"]].to_string(index=False))
b0 = loo.b.iloc[0]
worst = loo.iloc[1:].assign(d=lambda t: (t.b-b0).abs()).sort_values("d").iloc[-1]
print(f"\nLargest swing: {worst.label} moves b from {b0:.4f} to {worst.b:.4f} "
      f"({(worst.b-b0)/b0:+.0%})")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
cols = [C_TREATED if "Baseline" in l else (C_POLICY if "Lithuania" in l else C_CONTROL)
        for l in loo.label]
ax.barh(loo.label, loo.b, color=cols, alpha=.85, edgecolor="white")
ax.errorbar(loo.b, loo.label, xerr=1.96*loo.se, fmt="none", color="black",
            linewidth=1.4, capsize=4)
ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=.5)
ax.set_xlabel("TWFE coefficient with 95% CI (female homicide, per 100,000)")
ax.set_title("Leave-one-control-out — a single control country moves the estimate")
ax.grid(True, axis="x", alpha=.25)
fig_note("Design diagnostic, run before estimation. Source: UNODC UN-CTS.")
save_fig(OUT["figures"], "control_group_influence.png")

## 5. Statistical power

A null result is interpretable only against the effect size the design could
have detected. The minimum detectable effect at 80% power, expressed as a share
of the treated group's own baseline mean, is the single most important number
for reading the results in notebook 02.

In [ ]:
pw = []
for k in ["fhr", "male_hom", "log_ratio", "fem_ipf"]:
    s = SAMPLES[k]
    r = E.feols(s, k, ["did_interaction"], ["country","year"], cluster="country", name=k)
    base_mean = s.loc[s.treated_ever==1, k].mean()
    m = E.mde(r["did_interaction"]["se"], baseline=abs(base_mean) if base_mean else None)
    pw.append({"outcome": k, "label": OUTCOME_BY_KEY[k]["label"],
               "coef": r["did_interaction"]["coef"], "se": m["se"],
               "ci_low": r["did_interaction"]["ci_low"],
               "ci_high": r["did_interaction"]["ci_high"],
               "mde_80pct": m["mde"], "treated_mean": base_mean,
               "mde_as_pct_of_mean": m.get("mde_pct_of_baseline")})
pw_df = pd.DataFrame(pw)
pw_df.to_csv(OUT["diagnostics"] / "minimum_detectable_effects.csv", index=False)
print(pw_df.to_string(index=False))
print("\nRead this as: effects smaller than the MDE column CANNOT be detected by")
print("this design at 80% power. A non-significant result below that threshold is")
print("uninformative about whether an effect exists.")

## 6. Pre-treatment trends

Two tests. The joint test that all pre-treatment event-study coefficients are
zero has little power against a smooth differential trend, which is precisely
the alternative that threatens this design, so a linear pre-trend test is
reported alongside it.

Failing to reject parallel pre-trends is not evidence that they hold; it is the
absence of evidence that they do not.

In [ ]:
# Recorded sexual violence is analysed on the log transformation used for
# its reported regression, so its pre-trend diagnostic matches that model.
PRETREND_OUTCOME = {"svr": "log_svr"}

pt_rows = []
for k in ["fhr", "male_hom", "log_ratio", "fem_ipf", "svr"]:
    if not supports(k, "event_study"):
        continue
    y = PRETREND_OUTCOME.get(k, k)
    es, diag, res = E.event_study(SAMPLES[k], y)
    lin = E.linear_pretrend_test(SAMPLES[k], y)
    pre = es[es.kind == "pre"]
    pt_rows.append({"outcome": k, "label": OUTCOME_BY_KEY[k]["label"],
                    "n_pre_coefs": len(pre),
                    "pre_all_same_sign": diag.get("pre_coefs_all_same_sign"),
                    "slope_through_pre_coefs": diag.get("pre_trend_slope_through_coefs"),
                    "linear_pretrend_b": lin["b"], "linear_pretrend_se": lin["se"],
                    "linear_pretrend_p": lin["p"]})
    es.to_csv(OUT["diagnostics"] / f"pretrend_event_study_{OUTCOME_BY_KEY[k]['file_slug']}.csv", index=False)
pt = pd.DataFrame(pt_rows)
print(pt.to_string(index=False))
pt.to_csv(OUT["diagnostics"] / "pretrend_tests.csv", index=False)
print("\nNOTE: a non-significant linear pre-trend is NOT evidence for parallel")
print("trends. Compare the pre-period confidence intervals in the event-study")
print("tables against the size of the effect being estimated.")

In [ ]:
# Does the event-study path show a JUMP at treatment, or a continuing trend?
es, _, _ = E.event_study(SAMPLES["fhr"], "fhr")
pre = es[es.kind == "pre"]
sl, ic = np.polyfit(pre.event_time, pre.coef, 1)
post = es[es.kind == "post"].copy()
post["pretrend_extrapolation"] = ic + sl * post.event_time
post["excess_over_pretrend"] = post.coef - post.pretrend_extrapolation
print(f"Line through the pre-period coefficients: slope={sl:+.4f}/yr, intercept={ic:+.4f}")
print(post[["event_time","coef","pretrend_extrapolation","excess_over_pretrend"]].to_string(index=False))
print(f"\nCoefficient at t=0 is {es.loc[es.event_time==0,'coef'].iloc[0]:+.4f} "
      f"(t=-1 is 0 by construction): no discontinuity at ratification.")
post.to_csv(OUT["diagnostics"] / "pretrend_linear_vs_event_study.csv", index=False)

## 7. Male homicide comparison: placebo treatment timing

Treatment is reassigned to four years before each country's actual ratification,
using pre-treatment data only. An effect estimated before treatment can be
attributed to the design rather than to the Convention, so an outcome that fails
this test cannot support a causal reading.

In [ ]:
plc = []
for k in ["fhr", "male_hom", "log_ratio", "svr", "rape",
          "wbl_dv_legislation", "wbl_femicide_law"]:
    s = SAMPLES[k]
    pre_only = s[s.post_ratification == 0].copy()
    pre_only["fake_post"] = ((pre_only.treated_ever == 1) &
                             (pre_only.year >= pre_only.convention_ratified_year - 4)).astype(float)
    if pre_only.fake_post.sum() < 10 or pre_only.fake_post.nunique() < 2:
        continue
    try:
        r = E.feols(pre_only, k, ["fake_post"], ["country","year"],
                    cluster="country", name=f"placebo {k}")
        rr = E.coef_row(r, "fake_post", OUTCOME_BY_KEY[k]["label"])
        rr["verdict"] = "FAILS (placebo effect detected)" if rr["p"] < 0.10 else "passes"
        plc.append(rr)
    except RuntimeError as e:
        plc.append({"label": OUTCOME_BY_KEY[k]["label"], "b": np.nan, "se": np.nan,
                    "p": np.nan, "verdict": f"not estimable: {e}"})
plc_df = pd.DataFrame(plc)
print(plc_df[["label","b","se","p","verdict"]].to_string(index=False))
plc_df.to_csv(OUT["diagnostics"] / "placebo_timing_tests.csv", index=False)

## 8. Synthetic-control feasibility

Asked before anything is fitted. A synthetic control is a convex combination of
donor countries, so it can only reproduce a treated unit whose outcome level
lies inside the range spanned by the donors. If every donor sits above the
treated country, no set of non-negative weights summing to one can reach it.

Donors must also be genuinely untreated: a country that ratifies during the
treated unit's post-treatment period is itself treated.

In [ ]:
ALL_YEARS = sorted([y for y in df.year.unique() if y >= 2001])
cands = [("Spain", 2014), ("Italy", 2013), ("Portugal", 2013), ("Austria", 2013),
         ("Serbia", 2013), ("Montenegro", 2013), ("Albania", 2013)]
feas = SCM.feasibility_report(df, cands, "fhr", ALL_YEARS, )
print(feas.to_string(index=False))
feas.to_csv(OUT["diagnostics"] / "scm_feasibility.csv", index=False)
print("\ninside_convex_hull_on_levels=False means NO convex combination of donors")
print("can reproduce that country's level. Synthetic control is not applicable.")

## 9. Permitted methods by outcome

Written to `outputs/diagnostics/outcome_method_permissions.csv`, which notebook
02 reads. The permissions come from the outcome registry in
`src/analysis_helpers.py`, where each restriction is justified against the
coverage and measurement properties recorded above. A method not listed for an
outcome is not estimated for it anywhere in the project.

In [ ]:
verdict = []
for o in OUTCOMES:
    s = SAMPLES[o["key"]]
    verdict.append({
        "outcome": o["key"], "label": o["label"], "role": o["role"],
        "N": len(s), "treated_countries": int(s.loc[s.treated_ever.eq(1),"country"].nunique()),
        "control_countries": int(s.loc[s.treated_ever.eq(0),"country"].nunique()),
        "methods_allowed": "|".join(o["methods"]),
        "scm_allowed": "country-level; see scm_feasibility.csv",
        "rationale": o["note"]})
v = pd.DataFrame(verdict)
v.to_csv(OUT["diagnostics"] / "outcome_method_permissions.csv", index=False)
print(v[["label","N","treated_countries","control_countries","methods_allowed"]].to_string(index=False))
print()
print("SCM feasibility is a COUNTRY-level property, not an outcome-level one.")
print("See section 8. Spain, Italy, Portugal and Austria lie OUTSIDE the convex")
print("hull of the uncontaminated donor pool: every donor sits above them, so no")
print("convex combination can reproduce their level. Serbia, Montenegro and")
print("Albania lie inside it, and are fitted in notebook 04, section 2b.")
software_manifest(OUT, DATA_PATH)
print("\nDiagnostics complete.")

## 10. Consolidated design diagnostics

`outputs/diagnostics/design_diagnostics.csv` collects the checks that bear on
whether the design is credible, so that the methods chapter can cite one table
rather than seven.

In [ ]:
diag = []


def add(check, outcome, statistic, value, reading):
    diag.append({"check": check, "outcome": outcome, "statistic": statistic,
                 "value": value, "reading": reading})


for _, r in pw_df.iterrows():
    add("Statistical power", r["label"], "MDE as share of treated mean",
        round(float(r["mde_as_pct_of_mean"]), 4),
        "Effects smaller than this could not have been detected")

for _, r in pt.iterrows():
    p_lin = float(r["linear_pretrend_p"])
    add("Linear pre-treatment trend", r["label"], "p-value", round(p_lin, 4),
        "Not rejected" if p_lin > 0.10 else "Rejected: differential pre-trend")

for _, r in plc_df.iterrows():
    add("Placebo treatment timing", r["label"], "p-value",
        round(float(r["p"]), 4), r["verdict"])

loo_ctrl = pd.read_csv(OUT["diagnostics"] / "leave_one_control_out.csv")
base = float(loo_ctrl.loc[loo_ctrl.label.str.contains("All", case=False), "b"].iloc[0]) \
    if loo_ctrl.label.str.contains("All", case=False).any() else float(loo_ctrl["b"].iloc[0])
spread = float(loo_ctrl["b"].max() - loo_ctrl["b"].min())
add("Control-group influence", "Female homicide",
    "range of estimate across leave-one-control-out", round(spread, 4),
    f"Baseline {base:+.4f}; one control country moves it by up to "
    f"{spread:.4f}")

for _, r in feas.iterrows():
    add("Synthetic-control feasibility", r["country"], "donors below treated",
        int(r["donors_below_treated"]),
        "Inside donor hull" if bool(r["inside_convex_hull_on_levels"])
        else "Outside donor hull; synthetic control not applicable")

design = export_table(diag, OUT["diagnostics"] / "design_diagnostics.csv",
                      "design diagnostics")
print(design.to_string(index=False))